# Session 2 · Day 1 - Build Across Data Sources

Real projects rarely start from one clean file. The institution registry arrives as a Parquet
export, the accreditation list as a government CSV, the program compendium as an Excel workbook, and
boundaries as a GeoPackage or a shapefile. This notebook loads the **real TESDA datasets** from each
of those source types, then lands the institution list in **one typed schema** ready for the rest of
Session 2.

### What this notebook covers
1. Load the real institution list from **Parquet**
2. Load the real assessment-center and program files from **CSV / Excel**
3. Round-trip a table through a **SQL** database (SQLite) and read a join back
4. Load geometry from **GeoJSON** and a **GeoPackage**
5. Load a **shapefile** (and meet its 10-character field-name limit)
6. Unify the institution list into **one typed schema**
7. Save the unified table as CSV, GeoJSON, and shapefile
8. Query the SQL source with a region and a qualification-count filter *(exercise)*

> **How to use this notebook**
> - ▶ **Run each cell in order** (`Shift+Enter`). Later cells depend on earlier ones.
> - ✏️ **Exercise cells** have `### START CODE HERE ###` … `### END CODE HERE ###`. Write
>   code between those lines only.
> - ✅ After an exercise, run the **check cell**. Fix the code until it prints a pass.
> - 🎛️ **Play-around cells** are open to tweak, change a value, re-run, see what happens.
>
> _Real data: every table loads from the public TESDA bucket over the network, so an internet
> connection is required. The large boundary files are read in place, so only the features actually
> needed travel over the wire._

### New to Python or pandas?

Anyone coming from another language can run `S2_commands_primer` first. It lays out the mental model in one page: a `DataFrame` is a table (a SQL table, an R `data.frame`, a spreadsheet), a `merge` is a `JOIN`, `NaN` is the missing value, and a chain of `.method().method()` reads left to right. The comments in the code below point out each pandas idiom as it appears.

### Choose a format

Each format trades off differently, so the right choice depends on what the data carries and where it is going.

- **CSV**: simple and universal. Any tool reads it, but it stores no geometry and no types, so every value arrives as text until parsed.
- **Excel**: carries multiple sheets and light formatting, convenient for hand-maintained tables, but slow to read and awkward for large or automated pipelines.
- **Parquet**: columnar and compressed, fast to read and type-aware, efficient for wide analytical tables when only a few columns are needed.
- **GeoJSON**: carries geometry and serves cleanly to web maps, at the cost of large, verbose files. It is always geographic (EPSG:4326).
- **GeoPackage**: a single SQLite file that holds many vector layers with full types and indexes, a modern replacement for the shapefile bundle.
- **Shapefile**: the GIS standard every desktop tool opens, but it is really a bundle of files (`.shp`, `.shx`, `.dbf`, `.prj`, `.cpg`) and it truncates field names to 10 characters.
- **SQL database**: relational and queryable, and it scales to large tables through indexes and joins, at the cost of running a database.

As a rule of thumb, reach for Parquet or CSV for plain tables, GeoPackage or GeoJSON when geometry travels with the data, shapefile only when a tool demands it, and SQL when the data is large or shared across many users.

In [ ]:
# Run once per Colab session (skip if already installed locally).
!pip -q install geopandas shapely pyproj fiona pyogrio openpyxl sqlalchemy

In [ ]:
import os
import tempfile
import sqlite3
from urllib.parse import quote
import numpy as np
import pandas as pd
import geopandas as gpd
pd.set_option("display.max_columns", 30)

# Public TESDA bucket. Spaces in a filename have to be percent-encoded for an HTTP read.
BASE = "https://storage.googleapis.com/tesda-datasets/data/"
def source(name):
    return BASE + quote(name)

# The boundary files are hundreds of megabytes. GDAL's /vsicurl/ driver opens them in place over
# HTTP and fetches only the byte ranges a read touches, so a few rows cost a few requests.
VSI = "/vsicurl/" + BASE
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".gpkg,.shp,.shx,.dbf,.prj,.cpg"
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"

# One temporary folder for every file written and read back in this notebook.
WORK = tempfile.mkdtemp(prefix="s2d1_00_")
print("working folder:", WORK)

In [ ]:
# ------------------------------------------------------------------
# Optional: load from Google Drive instead of the public bucket.
# For anyone who already has the TESDA dataset in their Google Drive.
# Uncomment, mount the Drive, and point BASE at the folder with the files.
# With a local BASE, read a file directly as BASE + "<name>" using real
# spaces in the name. The source() helper (which turns spaces into %20)
# and the "/vsicurl/" prefix are only needed for the HTTP bucket.
# ------------------------------------------------------------------
# from google.colab import drive
# drive.mount("/content/drive")
# BASE = "/content/drive/MyDrive/TESDA/data/"   # <- the dataset folder (end with a slash)

## Load the institution list (Parquet)

The backbone of Session 2 is one table of TESDA training providers and assessment centers, with
coordinates and PSGC codes already checked: `tesda_coordinates.parquet`. Parquet stores column types,
so `pd.read_parquet` returns a typed frame straight away with no dtype guessing.

In [ ]:
inst = pd.read_parquet(source("tesda_coordinates.parquet"))
print("institutions:", inst.shape)
inst.head(3)

**Expected output:** `institutions: (8007, 21)`, then the first three institution rows. Each row is
one institution as recorded in one source file, carrying a `tesda_inst_id`, a name, a PSGC-aligned
address, coordinates, a `coord_status` flag, and `institution_classification` (provider, assessment
center, or both).

## Load CSV and Excel

Two more government files describe what each institution offers. The **Accredited Assessment
Centers** list is a CSV. The **Compendium of Registered Programs** is an Excel workbook with several
sheets. Both read with one pandas call. Reading every column as text (`dtype=str`) avoids silent
type coercion of ids and codes, `encoding="utf-8-sig"` strips the byte-order mark some exports carry,
and the personal-data columns are dropped at the door.

In [ ]:
ac = pd.read_csv(
    source("2026 Accredited Compentency Assessment Center based on July 15, 2026 database.csv"),
    dtype=str, encoding="utf-8-sig",   # read every value as text; utf-8-sig drops the BOM some exports add
)
ac = ac.apply(lambda s: s.str.strip())              # apply the trim to every column; .str runs a string op down a column
ac = ac.drop(columns=["Center Manager", "Phone"])   # personal data: never kept
print("read", ac.shape[0], "accreditation rows from CSV")
ac.head(3)

**Expected output:** `read 12871 accreditation rows from CSV`, then the first three rows. One row is
one accreditation: a center may assess many qualifications, so it appears on many rows.

In [ ]:
comp = pd.read_excel(
    source("Compendium of Registered Programs as of June 2026.xlsx"),
    sheet_name="Compendium ao June 2026", dtype=str,   # read_excel needs the openpyxl engine for .xlsx
)
comp = comp.drop(columns=["TEL. NO.", "E-mail Address", "Name of Trainer/s", "NTTC No."])
print("read", comp.shape[0], "registered-program rows from Excel")
comp.head(3)

**Expected output:** `read 21967 registered-program rows from Excel`, then the first three rows.
Passing `sheet_name` is what singles out the data sheet from the workbook's reference and summary
sheets.

## Load from a SQL database

Partner systems often expose a database instead of files. The accreditation rows loaded above go into
an **in-memory SQLite** database, built with the standard-library `sqlite3` module, across two tables:
a `centers` summary and the long `accreditations` table. The AC file carries no institution id, so a
surrogate `center_id` groups its rows by name and city. `pd.read_sql_query` reads a query back.

In [ ]:
# Surrogate key: the AC file has no institution id, so group rows by a normalized name + city.
def _key(s):
    # fill blanks, lower-case, and drop everything but letters and digits, so spacing
    # and punctuation stop splitting the same center into two groups
    return s.fillna("").str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
# factorize assigns each distinct "name|city" key its own integer id; [0] takes those codes
ac["center_id"] = pd.factorize(_key(ac["Assessment Center"]) + "|" + _key(ac["City Name"]))[0]

# One row per center, with a count of the qualifications it may assess.
centers = (ac.groupby("center_id")
             .agg(center_name=("Assessment Center", "first"),
                  region=("Region Name", "first"),
                  province=("Province Name", "first"),
                  city=("City Name", "first"),
                  n_quals=("Qualification", "count"))
             .reset_index())

# One row per (center, qualification): the long accreditations table.
accreditations = (ac[["center_id", "Qualification", "Sector Title", "Date of Expiry"]]
                  .rename(columns={"Qualification": "qualification",
                                   "Sector Title": "sector",
                                   "Date of Expiry": "expires_on"}))

conn = sqlite3.connect(":memory:")                  # in-memory database, no file needed
centers.to_sql("centers", conn, index=False, if_exists="replace")
accreditations.to_sql("accreditations", conn, index=False, if_exists="replace")

centers_sql = pd.read_sql_query("SELECT * FROM centers", conn)
print("read", centers_sql.shape[0], "centers from SQL; accreditations table has",
      pd.read_sql_query("SELECT COUNT(*) AS n FROM accreditations", conn)["n"][0], "rows")
centers_sql.head()

**Expected output:** `read 2712 centers from SQL; accreditations table has 12871 rows`, then the
first five center rows.

Many teams prefer **SQLAlchemy** for a uniform connection interface across database engines. The
same read through an engine looks like this, swap the URL for `postgresql://…` or `mysql://…` in
production:

In [ ]:
from sqlalchemy import create_engine

engine = create_engine("sqlite://")                 # in-memory; use a real URL in production
centers.to_sql("centers", engine, index=False, if_exists="replace")
with engine.connect() as cx:
    centers_sa = pd.read_sql_query("SELECT center_id, region, n_quals FROM centers", cx)
print("read", centers_sa.shape, "via SQLAlchemy engine")
centers_sa.head()

## Load GeoJSON and a GeoPackage

Location data arrives as a vector format. Point geometry attaches to the institution pins (only the
rows flagged `valid`), geopandas writes them to **GeoJSON**, and `gpd.read_file` reads them back.
GeoJSON is always in geographic coordinates (**EPSG:4326**), so the CRS survives the round trip.

In [ ]:
valid = inst["coord_status"] == "valid"
inst_gdf = gpd.GeoDataFrame(
    inst.copy(),
    # points_from_xy builds a point column from lon/lat; .where(valid) blanks the bad pins to NaN
    geometry=gpd.points_from_xy(inst["longitude"].where(valid), inst["latitude"].where(valid)),
    crs="EPSG:4326",
)
geojson_path = os.path.join(WORK, "tesda_institutions.geojson")
inst_gdf[valid].to_file(geojson_path, driver="GeoJSON")   # only the located pins

inst_geojson = gpd.read_file(geojson_path)
print("read", inst_geojson.shape[0], "features from GeoJSON; CRS:", inst_geojson.crs)
inst_geojson[["tesda_inst_id", "name", "geometry"]].head()

**Expected output:** `read 7891 features from GeoJSON; CRS: EPSG:4326` (the valid-pin count), then
three columns with `POINT (lon lat)` geometries.

The barangay boundaries ship as a **GeoPackage**: a single SQLite file that can hold many vector
layers with full types. The real file is about 300 MB, so reading it in place with `rows=` pulls only
the first features over the network, enough to inspect the schema without downloading everything.

In [ ]:
brgy_gpkg = gpd.read_file(VSI + "consolidated_geodata_matched.gpkg", rows=10)
print("read", brgy_gpkg.shape[0], "barangay boundaries from the GeoPackage; CRS:", brgy_gpkg.crs)
brgy_gpkg[["name", "adm1_en", "adm2_en", "adm3_en", "geometry"]].head()

**Expected output:** `read 10 barangay boundaries from the GeoPackage; CRS: EPSG:4326`, then a few
rows of named barangay polygons with their region, province, and municipality.

## Load a shapefile (`.shp`)

Government GIS data is very often shipped as an ESRI **shapefile**. A shapefile is really several
files sharing a name (`.shp`, `.shx`, `.dbf`, `.prj`, `.cpg`). A reader needs every sibling present.
Reading the remote file in place fetches only the pieces the first rows touch.

In [ ]:
brgy_shp = gpd.read_file(VSI + "shapefiles/phl_admbnda_adm4_updated.shp", rows=10)
print("read", brgy_shp.shape[0], "barangay boundaries from the shapefile; CRS:", brgy_shp.crs)
print("columns:", list(brgy_shp.columns)[:6], "...")

**Expected output:** `read 10 barangay boundaries from the shapefile; CRS: EPSG:4326`, then the first
few column names (`ADM4_EN`, `ADM4_PCODE`, …), each already 10 characters or fewer.

The format has one real gotcha: it truncates every field name to **10 characters**. Writing the
institution table out to a shapefile and reading it back shows it: `institution_classification`
(26 characters) comes back as `institutio`. Always check the columns after reading a shapefile, and
rename them back before joining.

In [ ]:
shp_path = os.path.join(WORK, "tesda_institutions.shp")
demo_cols = ["tesda_inst_id", "region", "institution_classification",
             "type_of_institution", "psgc_observed_barangay", "geometry"]
inst_gdf.loc[valid, demo_cols].to_file(shp_path)   # ESRI Shapefile is the default driver

inst_shp = gpd.read_file(shp_path)
print("columns after shapefile round trip:", list(inst_shp.columns))

# Rename every truncated name back to its schema name before joining.
inst_shp = inst_shp.rename(columns={
    "tesda_inst": "tesda_inst_id",
    "institutio": "institution_classification",
    "type_of_in": "type_of_institution",
    "psgc_obser": "psgc_observed_barangay",
})
print("after rename:", list(inst_shp.columns))

**Expected output:** the column list shows truncated names (`tesda_inst`, `institutio`,
`type_of_in`, `psgc_obser`), and after the rename all four are back to their schema names (`tesda_inst_id`,
`institution_classification`, `type_of_institution`, `psgc_observed_barangay`).

## Unify to one typed schema

Each source produced records in a different shape. One **typed schema** settles the institution list
so downstream code never has to guess a dtype. The needed columns are cast explicitly, ready for the
PSGC alignment step that follows in `S2_D1_02`.

In [ ]:
SCHEMA = {
    "tesda_inst_id":              "string",
    "name":                       "string",
    "region":                     "string",
    "province":                   "string",
    "institution_classification": "string",
    "latitude":                   "float64",
    "longitude":                  "float64",
    "coord_status":               "string",
}
institutions = inst[list(SCHEMA)].astype(SCHEMA)
print("unified institutions:", institutions.shape)
institutions.dtypes

**Expected output:** `unified institutions: (8007, 8)`, then a dtypes listing where the ids, text,
and flag columns are `string` and `latitude`/`longitude` are `float64`.

### Save the unified table three ways

One typed table, three portable formats. `institutions` is written to a **CSV** for spreadsheets and
plain tabular tools, to **GeoJSON** for web maps and geographic exchange, and to an ESRI **shapefile**
for classic GIS software. Only the valid pins carry geometry, so the two vector files hold those rows.
All three land in the temporary working folder.

In [ ]:
# Attach point geometry to the valid pins so the two vector formats have coordinates to write.
_string_cols = {c: "object" for c in institutions.columns if institutions[c].dtype == "string"}
_valid = institutions["coord_status"] == "valid"
institutions_gdf = gpd.GeoDataFrame(
    institutions.astype(_string_cols),
    geometry=gpd.points_from_xy(institutions["longitude"].where(_valid), institutions["latitude"].where(_valid)),
    crs="EPSG:4326",
)
_located = institutions_gdf[_valid]   # only the located pins carry geometry

csv_out     = os.path.join(WORK, "institutions_unified.csv")
geojson_out = os.path.join(WORK, "institutions_unified.geojson")
shp_out     = os.path.join(WORK, "institutions_unified.shp")

institutions.to_csv(csv_out, index=False)                   # CSV (all rows)
_located.to_file(geojson_out, driver="GeoJSON")             # GeoJSON (EPSG:4326, valid pins)
_located.to_file(shp_out, driver="ESRI Shapefile")          # Shapefile (10-char names, valid pins)

print("CSV:      ", csv_out)
print("GeoJSON:  ", geojson_out)
print("Shapefile:", shp_out)
assert os.path.exists(csv_out), "CSV was not written"
assert os.path.exists(geojson_out), "GeoJSON was not written"
assert os.path.exists(shp_out), "Shapefile was not written"
print("All three files written.")

## ✏️ Exercise - query the SQL source

Write a SQL query, run it against the in-memory database from the SQL section, and store the result in
`result`. The query must:

1. return centers whose `region` equals `REGION` **and** whose `n_quals` is greater than `THRESH`,
2. **join** the `centers` table to the `accreditations` table on `center_id`,
3. select the columns `center_id`, `center_name`, and `qualification`.

Use the two placeholders (`?`) with `params=(REGION, THRESH)` so the query stays parameterized.
Complete the query string between the markers.

In [ ]:
REGION = "National Capital Region (NCR)"
THRESH = 15

### START CODE HERE ###   (write the SQL between the triple quotes)
query = """

"""
### END CODE HERE ###

result = pd.read_sql_query(query, conn, params=(REGION, THRESH))
print("rows returned:", len(result))
result.head()

## ✅ Check the work

Run this. It builds the same answer with pandas and compares it to the SQL result above.

In [ ]:
try:
    # pandas reference for the SAME filter and join.
    _busy = centers[(centers["region"] == REGION) & (centers["n_quals"] > THRESH)]["center_id"]
    ref = (accreditations[accreditations["center_id"].isin(_busy)]
           [["center_id", "qualification"]]
           .sort_values(["center_id", "qualification"]).reset_index(drop=True))

    assert result is not None, "result is None - did you write the query?"
    got = (result[["center_id", "qualification"]]
           .sort_values(["center_id", "qualification"]).reset_index(drop=True))
    assert len(got) == len(ref), f"expected {len(ref)} rows, got {len(got)}"
    assert list(got["center_id"]) == list(ref["center_id"]), "center_id set does not match"
    assert list(got["qualification"]) == list(ref["qualification"]), "qualification values do not match"
    print(f"✅ Passed! The query returned the {len(ref)} expected rows for {REGION}",
          f"with more than {THRESH} qualifications.")
except AssertionError as e:
    print("❌ Not yet:", e)

### ❓ Question

The SQL source held two tables: `centers` and the long `accreditations` table, one row for each
center-qualification pair. How many rows are in the `accreditations` table?

In [ ]:
q_accred_rows = None   # <- answer here (an integer)

In [ ]:
_accred_rows = int(pd.read_sql_query("SELECT COUNT(*) AS n FROM accreditations", conn)["n"][0])
if q_accred_rows == _accred_rows:
    print(f"✅ Correct! The accreditations table has {_accred_rows} rows.")
else:
    print(f"❌ Not yet: expected {_accred_rows}, got {q_accred_rows}.")

### ❓ Question

After the unify step, `institutions` carries the typed institution schema. How many columns does it
have?

In [ ]:
q_schema_cols = None   # <- answer here (an integer)

In [ ]:
_schema_cols = institutions.shape[1]
if q_schema_cols == _schema_cols:
    print(f"✅ Correct! The unified institution schema has {_schema_cols} columns.")
else:
    print(f"❌ Not yet: expected {_schema_cols}, got {q_schema_cols}.")

## 🎛️ Play around - change the filter

The region and the qualification-count threshold are adjustable. Edit the two values and re-run to
see how many center-qualification rows come back. Try a quieter region or a higher bar.

In [ ]:
region_pick = "Region VII - Central Visayas"   # try other "Region Name" values from the CSV
qual_min    = 20                               # try 10, 15, 30

q = """
SELECT c.center_id, c.center_name, c.n_quals, a.qualification
FROM centers c
JOIN accreditations a ON c.center_id = a.center_id
WHERE c.region = ? AND c.n_quals > ?
ORDER BY c.n_quals DESC
"""
out = pd.read_sql_query(q, conn, params=(region_pick, qual_min))
print(f"{region_pick} with more than {qual_min} qualifications: {len(out)} center-qualification rows")
out.head()

## Recap
- The same TESDA records arrive as Parquet, CSV, Excel, SQL, GeoJSON, GeoPackage, and shapefile, each
  with its own reader.
- Watch the **format gotchas**: shapefiles truncate field names to 10 characters, GeoJSON is always
  EPSG:4326, and reading every column of a messy CSV as text avoids silent coercion.
- Hundreds of megabytes of boundaries need not download in full: `/vsicurl/` reads a remote file in
  place and fetches only the bytes a query touches.
- Land every source in **one typed schema** so downstream code never guesses a dtype.

**Next:** `S2_D1_01`: framing the analytical dataset and attaching geometry with an explicit CRS.

## References & further reading

Documentation and background for the tools and methods in this notebook.

- [GeoPandas, user guide & API](https://geopandas.org/)
- [GeoPandas, reading & writing files (SHP/GeoJSON/GeoPackage)](https://geopandas.org/en/stable/docs/user_guide/io.html)
- [Shapely (geometry)](https://shapely.readthedocs.io/)
- [pyproj / CRS (EPSG:4326)](https://pyproj4.github.io/pyproj/stable/)
- [pandas, user guide](https://pandas.pydata.org/docs/user_guide/)
- [pandas IO (read_csv / read_excel / read_parquet)](https://pandas.pydata.org/docs/user_guide/io.html)
- [Apache Parquet](https://parquet.apache.org/)
- [Python sqlite3](https://docs.python.org/3/library/sqlite3.html)
- [SQLAlchemy](https://www.sqlalchemy.org/)
- [GeoJSON, RFC 7946](https://datatracker.ietf.org/doc/html/rfc7946)
- [OGC GeoPackage](https://www.geopackage.org/)
- [GDAL /vsicurl/ (reading remote files in place)](https://gdal.org/user/virtual_file_systems.html#vsicurl-http-https-ftp-files-random-access)
- [PSA, Philippine Standard Geographic Code (PSGC)](https://psa.gov.ph/classification/psgc)